# Edge TTS — Arabic / Egyptian voice test

Microsoft Edge TTS only. TTS input is **only** mapped pronunciation + period (or phrase text + period).

## 1. Dependencies

In [1]:
import subprocess
import sys

def _ensure(pkg: str, import_name: str | None = None):
    name = import_name or pkg
    try:
        __import__(name)
        print(f"[deps OK] {pkg}")
        return True
    except ImportError:
        print(f"[deps] installing {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        __import__(name)
        print(f"[deps OK] {pkg} (installed)")
        return True

_ensure("edge_tts")
_ensure("nest_asyncio")
_ensure("soundfile")
_ensure("librosa")
_ensure("numpy")
_ensure("IPython", "IPython")
print("Edge TTS dependency bootstrap done.")


[deps OK] edge_tts
[deps OK] nest_asyncio
[deps OK] soundfile
[deps OK] librosa


[deps OK] numpy
[deps OK] IPython
Edge TTS dependency bootstrap done.


## 2. Paths and output folders

In [2]:
from pathlib import Path

BASE_DIR = Path.cwd().resolve()
AUDIO_ROOT = BASE_DIR / "generated_audio"
DIR_ALPHABET = AUDIO_ROOT / "edge_tts_alphabet"
DIR_NUMBERS = AUDIO_ROOT / "edge_tts_numbers"
DIR_PHRASES = AUDIO_ROOT / "edge_tts_phrases"

for d in (DIR_ALPHABET, DIR_NUMBERS, DIR_PHRASES):
    d.mkdir(parents=True, exist_ok=True)

print("cwd:", BASE_DIR)
print("alphabet:", DIR_ALPHABET)
print("numbers:", DIR_NUMBERS)
print("phrases:", DIR_PHRASES)


cwd: C:\Users\sondo\Desktop\Voice Model Stuff
alphabet: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_alphabet
numbers: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_numbers
phrases: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_phrases


## 3. Egyptian letter, number, and phrase mappings

In [ ]:
LETTER_PRON = {
    "ا": "أَلِف", "أ": "أَلِف", "ب": "بِه", "ت": "تِه", "ث": "سِه",
    "ج": "جِيم", "ح": "حَا", "خ": "خَا", "د": "دَال", "ذ": "ذَال",
    "ر": "رَا", "ز": "زَا", "س": "سِين", "ش": "شِين", "ص": "صَاد",
    "ض": "ضَاد", "ط": "طَا", "ظ": "ظَا", "ع": "عِين", "غ": "غِين",
    "ف": "فَا", "ق": "قَاف", "ك": "كَاف", "ل": "لَام", "م": "مِيم",
    "ن": "نُون", "ه": "هَا", "و": "وَاو", "ي": "يَا",
}
LETTERS_ORDER = list("اأبتثجحخدذرزسشصضطظعغفقكلمنهوي")
NUMBER_PRON = {
    0: "صِفْر", 1: "وَاحِد", 2: "اِتْنِين", 3: "تَلَاتَه", 4: "أَرْبَعَه",
    5: "خَمْسَه", 6: "سِتَّه", 7: "سَبْعَه", 8: "تَمَانْيَه", 9: "تِسْعَه", 10: "عَشَرَه",
}
PHRASES = [
    ("ana_basme3ak", "أَنَا بَسْمَعَك."),
    ("mama", "مَامَا."),
    ("beit", "بَيْت."),
    ("khamsa", "خَمْسَه."),
    ("itneen", "اِتْنِين."),
    ("ahlan_beek", "أَهْلًا بِيك."),
    ("ezzayak", "إِزَّيَّك."),
    ("ayez_arooh_elbeit", "عَايِز أَرُوح البَيْت."),
]

def with_period(text: str) -> str:
    t = text.strip()
    return t if t.endswith(".") else t + "."

def letter_tts(letter: str) -> str:
    return with_period(LETTER_PRON.get(letter, letter))

def number_tts(n: int) -> str:
    return with_period(NUMBER_PRON[n])

LETTER_FILE_SLUG = {
    "ا": "alif", "أ": "alif_hamza", "ب": "beh", "ت": "teh", "ث": "seh",
    "ج": "geem", "ح": "ha", "خ": "kha", "د": "dal", "ذ": "zal",
    "ر": "ra", "ز": "za", "س": "seen", "ش": "sheen", "ص": "sad",
    "ض": "dad", "ط": "ta", "ظ": "dha", "ع": "ein", "غ": "ghein",
    "ف": "fa", "ق": "qaf", "ك": "kaf", "ل": "lam", "م": "meem",
    "ن": "noon", "ه": "ha_end", "و": "waw", "ي": "ya",
}

def letter_file_stem(letter: str) -> str:
    """Readable filename: letter + Arabic char + pronunciation slug."""
    slug = LETTER_FILE_SLUG.get(letter, f"u{ord(letter):04x}")
    return f"letter_{letter}_{slug}"



## 4. Select Edge TTS voice

In [4]:
import asyncio

import edge_tts
import nest_asyncio

nest_asyncio.apply()

PREFERRED_VOICES = ["ar-EG-SalmaNeural", "ar-EG-ShakirNeural"]
EDGE_VOICE = None
EDGE_TTS_LOADED = False
_ALL_VOICES = None

async def _list_voices():
    global _ALL_VOICES
    if _ALL_VOICES is None:
        _ALL_VOICES = await edge_tts.list_voices()
    return _ALL_VOICES

async def select_edge_voice() -> str:
    voices = await _list_voices()
    by_short = {v["ShortName"]: v for v in voices}

    for pref in PREFERRED_VOICES:
        if pref in by_short:
            return pref

    ar_names = sorted(
        v["ShortName"]
        for v in voices
        if str(v.get("Locale", "")).lower().startswith("ar")
    )
    print("Preferred voices unavailable. Available Arabic voices:")
    for name in ar_names:
        print(" ", name)

    eg = [
        v["ShortName"]
        for v in voices
        if "EG" in v.get("Locale", "") or "-EG-" in v.get("ShortName", "")
    ]
    if eg:
        return eg[0]
    if ar_names:
        return ar_names[0]
    raise RuntimeError("No Arabic Edge TTS voice found")

EDGE_VOICE = asyncio.get_event_loop().run_until_complete(select_edge_voice())
EDGE_TTS_LOADED = True
print(f"Selected Edge TTS voice: {EDGE_VOICE}")


Selected Edge TTS voice: ar-EG-SalmaNeural


## 5. Edge TTS synthesis, trim, and gibberish check helpers

In [5]:
import time
import wave
from typing import Optional

import numpy as np
from IPython.display import Audio as IPA, display

GENERATED = {"alphabet": [], "numbers": [], "phrases": []}
GIBBERISH_LOG = []
PLAYBACK_SHOWN = False
TRIM_BACKEND = None

MAX_DUR_LETTER = 2.5
MAX_DUR_NUMBER = 3.0
MAX_DUR_PHRASE = 8.0
TRIM_THRESHOLD_S = 0.05

def _init_trim_backend():
    global TRIM_BACKEND
    if TRIM_BACKEND is not None:
        return TRIM_BACKEND
    try:
        import librosa  # noqa: F401
        TRIM_BACKEND = "librosa"
    except ImportError:
        TRIM_BACKEND = "soundfile"
    print(f"[trim] using {TRIM_BACKEND}")
    return TRIM_BACKEND

def _read_audio(path: Path):
    backend = _init_trim_backend()
    if backend == "librosa":
        import librosa
        y, sr = librosa.load(str(path), sr=None, mono=True)
        return y.astype(np.float32), int(sr)
    import soundfile as sf
    y, sr = sf.read(str(path), dtype="float32", always_2d=False)
    if getattr(y, "ndim", 1) > 1:
        y = y.mean(axis=1)
    return y.astype(np.float32), int(sr)

def _write_wav(path: Path, y: np.ndarray, sr: int):
    import soundfile as sf
    sf.write(str(path), np.clip(y, -1.0, 1.0), sr)

def audio_duration(path: Path) -> float:
    y, sr = _read_audio(path)
    return len(y) / sr if sr else 0.0

def _simple_amplitude_trim(y: np.ndarray, sr: int, top_db: float = 25.0):
    if y.size == 0:
        return y
    frame = max(1, int(sr * 0.01))
    hop = frame
    n_frames = max(1, 1 + (len(y) - frame) // hop)
    rms = []
    for i in range(n_frames):
        chunk = y[i * hop : i * hop + frame]
        rms.append(float(np.sqrt(np.mean(chunk * chunk) + 1e-12)))
    rms = np.array(rms)
    peak = float(rms.max()) if rms.size else 0.0
    if peak <= 0:
        return y[: max(1, int(sr * 0.15))]
    thresh = peak * (10 ** (-top_db / 20.0))
    voiced = np.where(rms >= thresh)[0]
    if voiced.size == 0:
        return y[: max(1, int(sr * 0.15))]
    start = int(voiced[0] * hop)
    end = min(len(y), int((voiced[-1] + 1) * hop + frame))
    pad = int(sr * 0.03)
    return y[max(0, start - pad) : min(len(y), end + pad)]

def trim_audio_array(y: np.ndarray, sr: int, max_duration: float):
    backend = _init_trim_backend()
    if backend == "librosa":
        import librosa
        yt, _ = librosa.effects.trim(y, top_db=25)
        if yt.size == 0:
            yt = y
    else:
        yt = _simple_amplitude_trim(y, sr)
    max_samples = int(max_duration * sr)
    if yt.size > max_samples:
        yt = yt[:max_samples]
    return yt

async def synth_edge_mp3(text: str, mp3_path: Path) -> float:
    mp3_path.parent.mkdir(parents=True, exist_ok=True)
    t0 = time.perf_counter()
    comm = edge_tts.Communicate(text, EDGE_VOICE)
    await comm.save(str(mp3_path))
    return time.perf_counter() - t0

def mp3_to_trimmed_wav(mp3_path: Path, wav_path: Path, max_duration: float) -> tuple[float, float, bool]:
    raw_dur = audio_duration(mp3_path)
    y, sr = _read_audio(mp3_path)
    yt = trim_audio_array(y, sr, max_duration)
    _write_wav(wav_path, yt, sr)
    clean_dur = len(yt) / sr
    trimmed = (raw_dur - clean_dur) > TRIM_THRESHOLD_S
    return raw_dur, clean_dur, trimmed

def format_bytes(nb: int) -> str:
    if nb < 1024:
        return f"{nb} B"
    if nb < 1024**2:
        return f"{nb / 1024:.2f} KB"
    return f"{nb / 1024 / 1024:.2f} MB"

async def generate_clean_audio(
    label: str,
    tts_text: str,
    wav_path: Path,
    mp3_path: Path,
    max_duration: float,
    kind: str,
    show_playback: bool = False,
) -> Optional[str]:
    global PLAYBACK_SHOWN

    print("original:", label)
    print("final text sent to Edge TTS:", tts_text)

    infer_s = await synth_edge_mp3(tts_text, mp3_path)
    if not mp3_path.is_file():
        print("FAILED mp3:", mp3_path)
        print("---")
        return None

    raw_dur, clean_dur, trimmed = mp3_to_trimmed_wav(mp3_path, wav_path, max_duration)
    saved = wav_path.resolve()

    print(f"raw audio duration: {raw_dur:.3f}s")
    print(f"cleaned audio duration: {clean_dur:.3f}s")
    print(f"trimming occurred: {trimmed}")
    print(f"mp3: {mp3_path.resolve()}")
    print(f"saved cleaned wav: {saved}")
    print(f"inference: {infer_s:.3f}s | mp3 {format_bytes(mp3_path.stat().st_size)} | wav {format_bytes(wav_path.stat().st_size)}")

    GIBBERISH_LOG.append({
        "label": label,
        "kind": kind,
        "raw_s": raw_dur,
        "clean_s": clean_dur,
        "trimmed": trimmed,
        "path": str(saved),
    })

    if show_playback:
        display(IPA(filename=str(wav_path)))
        PLAYBACK_SHOWN = True

    print("---")
    return str(saved)

def run_async(coro):
    return asyncio.get_event_loop().run_until_complete(coro)


## 6. Alphabet tests

In [ ]:
MAX_PLAYBACK = 5
seen_letters = set()
ok_alpha = 0

for letter in LETTERS_ORDER:
    if letter in seen_letters:
        continue
    seen_letters.add(letter)
    tts_text = letter_tts(letter)
    stem = letter_file_stem(letter)
    wav_path = DIR_ALPHABET / f"{stem}.wav"
    mp3_path = DIR_ALPHABET / f"{stem}.mp3"
    saved = run_async(
        generate_clean_audio(
            label=letter,
            tts_text=tts_text,
            wav_path=wav_path,
            mp3_path=mp3_path,
            max_duration=MAX_DUR_LETTER,
            kind="alphabet",
            show_playback=(ok_alpha < MAX_PLAYBACK),
        )
    )
    if saved:
        ok_alpha += 1
        GENERATED["alphabet"].append(saved)

ALPHABET_OK = ok_alpha == len(seen_letters) and ok_alpha > 0
print(f"Alphabet: {ok_alpha}/{len(seen_letters)} files")


## 7. Number tests

In [7]:
ok_num = 0
num_playback = 0

for n in range(0, 11):
    tts_text = number_tts(n)
    slug = f"num_{n:02d}"
    wav_path = DIR_NUMBERS / f"{slug}.wav"
    mp3_path = DIR_NUMBERS / f"{slug}.mp3"
    saved = run_async(
        generate_clean_audio(
            label=str(n),
            tts_text=tts_text,
            wav_path=wav_path,
            mp3_path=mp3_path,
            max_duration=MAX_DUR_NUMBER,
            kind="numbers",
            show_playback=(num_playback < 3),
        )
    )
    if saved:
        ok_num += 1
        num_playback += 1
        GENERATED["numbers"].append(saved)

NUMBERS_OK = ok_num == 11
print(f"Numbers: {ok_num}/11 files")


original: 0
final text sent to Edge TTS: صِفْر.


raw audio duration: 1.416s
cleaned audio duration: 0.384s
trimming occurred: True
mp3: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_numbers\num_00.mp3
saved cleaned wav: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_numbers\num_00.wav
inference: 2.619s | mp3 8.30 KB | wav 18.04 KB


---
original: 1
final text sent to Edge TTS: وَاحِد.


raw audio duration: 1.536s
cleaned audio duration: 0.512s
trimming occurred: True
mp3: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_numbers\num_01.mp3
saved cleaned wav: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_numbers\num_01.wav
inference: 1.159s | mp3 9.00 KB | wav 24.04 KB


---
original: 2
final text sent to Edge TTS: اِتْنِين.


...

raw audio duration: 1.608s
cleaned audio duration: 0.512s
trimming occurred: True
mp3: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_numbers\num_06.mp3
saved cleaned wav: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_numbers\num_06.wav
inference: 1.069s | mp3 9.42 KB | wav 24.04 KB
---
original: 7
final text sent to Edge TTS: سَبْعَه.


raw audio duration: 1.656s
cleaned audio duration: 0.597s
trimming occurred: True
mp3: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_numbers\num_07.mp3
saved cleaned wav: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_numbers\num_07.wav
inference: 0.862s | mp3 9.70 KB | wav 28.04 KB
---
original: 8
final text sent to Edge TTS: تَمَانْيَه.


raw audio duration: 1.728s
cleaned audio duration: 0.597s
trimming occurred: True
mp3: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_numbers\num_08.mp3
saved cleaned wav: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_numbers\num_08.wav
inference: 0.924s | mp3 10.12 KB | wav 28.04 KB
---
original: 9
final text sent to Edge TTS: تِسْعَه.


raw audio duration: 1.608s
cleaned audio duration: 0.512s
trimming occurred: True
mp3: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_numbers\num_09.mp3
saved cleaned wav: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_numbers\num_09.wav
inference: 1.002s | mp3 9.42 KB | wav 24.04 KB
---
original: 10
final text sent to Edge TTS: عَشَرَه.


raw audio duration: 1.752s
cleaned audio duration: 0.683s
trimming occurred: True
mp3: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_numbers\num_10.mp3
saved cleaned wav: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_numbers\num_10.wav
inference: 0.897s | mp3 10.27 KB | wav 32.04 KB
---
Numbers: 11/11 files


## 8. Egyptian Arabic phrase tests

In [8]:
ok_phrase = 0
phrase_playback = 0

for slug, phrase in PHRASES:
    tts_text = with_period(phrase)
    wav_path = DIR_PHRASES / f"{slug}.wav"
    mp3_path = DIR_PHRASES / f"{slug}.mp3"
    saved = run_async(
        generate_clean_audio(
            label=phrase,
            tts_text=tts_text,
            wav_path=wav_path,
            mp3_path=mp3_path,
            max_duration=MAX_DUR_PHRASE,
            kind="phrases",
            show_playback=(phrase_playback < 3),
        )
    )
    if saved:
        ok_phrase += 1
        phrase_playback += 1
        GENERATED["phrases"].append(saved)

PHRASES_OK = ok_phrase == len(PHRASES)
print(f"Phrases: {ok_phrase}/{len(PHRASES)} files")


original: أَنَا بَسْمَعَك.
final text sent to Edge TTS: أَنَا بَسْمَعَك.


raw audio duration: 1.992s
cleaned audio duration: 0.789s
trimming occurred: True
mp3: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_phrases\ana_basme3ak.mp3
saved cleaned wav: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_phrases\ana_basme3ak.wav
inference: 1.129s | mp3 11.67 KB | wav 37.04 KB


---
original: مَامَا.
final text sent to Edge TTS: مَامَا.


raw audio duration: 1.488s
cleaned audio duration: 0.448s
trimming occurred: True
mp3: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_phrases\mama.mp3
saved cleaned wav: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_phrases\mama.wav
inference: 0.954s | mp3 8.72 KB | wav 21.04 KB


---
original: بَيْت.
final text sent to Edge TTS: بَيْت.


...

raw audio duration: 1.632s
cleaned audio duration: 0.448s
trimming occurred: True
mp3: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_phrases\khamsa.mp3
saved cleaned wav: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_phrases\khamsa.wav
inference: 0.805s | mp3 9.56 KB | wav 21.04 KB
---
original: اِتْنِين.
final text sent to Edge TTS: اِتْنِين.


raw audio duration: 1.704s
cleaned audio duration: 0.661s
trimming occurred: True
mp3: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_phrases\itneen.mp3
saved cleaned wav: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_phrases\itneen.wav
inference: 0.818s | mp3 9.98 KB | wav 31.04 KB
---
original: أَهْلًا بِيك.
final text sent to Edge TTS: أَهْلًا بِيك.


raw audio duration: 1.752s
cleaned audio duration: 0.619s
trimming occurred: True
mp3: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_phrases\ahlan_beek.mp3
saved cleaned wav: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_phrases\ahlan_beek.wav
inference: 0.622s | mp3 10.27 KB | wav 29.04 KB
---
original: إِزَّيَّك.
final text sent to Edge TTS: إِزَّيَّك.


raw audio duration: 1.728s
cleaned audio duration: 0.640s
trimming occurred: True
mp3: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_phrases\ezzayak.mp3
saved cleaned wav: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_phrases\ezzayak.wav
inference: 0.813s | mp3 10.12 KB | wav 30.04 KB
---
original: عَايِز أَرُوح البَيْت.
final text sent to Edge TTS: عَايِز أَرُوح البَيْت.


raw audio duration: 2.376s
cleaned audio duration: 1.280s
trimming occurred: True
mp3: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_phrases\ayez_arooh_elbeit.mp3
saved cleaned wav: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\edge_tts_phrases\ayez_arooh_elbeit.wav
inference: 0.992s | mp3 13.92 KB | wav 60.04 KB
---
Phrases: 8/8 files


## 9. Gibberish / trailing silence summary

In [9]:
trimmed_count = sum(1 for g in GIBBERISH_LOG if g["trimmed"])
long_tail = [g for g in GIBBERISH_LOG if (g["raw_s"] - g["clean_s"]) > 0.35]

print(f"Total clips: {len(GIBBERISH_LOG)}")
print(f"Clips trimmed (> {TRIM_THRESHOLD_S}s removed): {trimmed_count}")
print(f"Clips with large tail removed (>0.35s): {len(long_tail)}")

if long_tail:
    print("\nLargest tail reductions:")
    for g in sorted(long_tail, key=lambda x: x["raw_s"] - x["clean_s"], reverse=True)[:8]:
        delta = g["raw_s"] - g["clean_s"]
        print(f"  {g['kind']} {g['label']!r}: {g['raw_s']:.3f}s -> {g['clean_s']:.3f}s (Δ {delta:.3f}s)")

avg_clean = sum(g["clean_s"] for g in GIBBERISH_LOG) / max(1, len(GIBBERISH_LOG))
print(f"\nAverage cleaned duration: {avg_clean:.3f}s")
GIBBERISH_OK = len(GIBBERISH_LOG) > 0


Total clips: 48
Clips trimmed (> 0.05s removed): 48
Clips with large tail removed (>0.35s): 48

Largest tail reductions:
  phrases 'أَنَا بَسْمَعَك.': 1.992s -> 0.789s (Δ 1.203s)
  numbers '5': 1.632s -> 0.448s (Δ 1.184s)
  phrases 'خَمْسَه.': 1.632s -> 0.448s (Δ 1.184s)
  alphabet 'ك': 1.704s -> 0.555s (Δ 1.149s)
  numbers '3': 1.704s -> 0.555s (Δ 1.149s)
  alphabet 'ا': 1.440s -> 0.299s (Δ 1.141s)
  alphabet 'أ': 1.440s -> 0.299s (Δ 1.141s)
  phrases 'أَهْلًا بِيك.': 1.752s -> 0.619s (Δ 1.133s)

Average cleaned duration: 0.461s


## 10. Final verification

In [10]:
checks = [
    ("Edge TTS loaded", EDGE_TTS_LOADED and bool(EDGE_VOICE)),
    ("Alphabet audio generated", ALPHABET_OK),
    ("Number audio generated", NUMBERS_OK),
    ("Phrase audio generated", PHRASES_OK),
    ("Audio playback displayed", PLAYBACK_SHOWN),
    ("Gibberish check completed", GIBBERISH_OK),
]

for label, ok in checks:
    mark = "✅" if ok else "❌"
    print(f"{mark} {label}")

ALL_OK = all(ok for _, ok in checks)
print("\nSelected Edge TTS voice:", EDGE_VOICE)
print("Alphabet files:", len(GENERATED["alphabet"]))
print("Number files:", len(GENERATED["numbers"]))
print("Phrase files:", len(GENERATED["phrases"]))


✅ Edge TTS loaded
✅ Alphabet audio generated
✅ Number audio generated
✅ Phrase audio generated
✅ Audio playback displayed
✅ Gibberish check completed

Selected Edge TTS voice: ar-EG-SalmaNeural
Alphabet files: 29
Number files: 11
Phrase files: 8
